# Laboratoire SIG – version corrigée et détaillée

Ce notebook correspond à une **version enseignant** du laboratoire.  
Il contient :

- le **code complet** ;
- des **explications détaillées** sur le rôle de chaque commande ;
- des **commentaires pédagogiques** pour accompagner les étudiants débutants.

## Fichiers utilisés

- CSV : `data/cci_v6_2003226_ppz_takuvik_above_45n_v2.csv`
- shapefile : `data/WDPA_WDOECM_Feb2025_Public_555637925_shp-polygons.shp`

## Variables importantes du CSV

- `latitudeitude` : latitudeitude
- `longitudegitude` : longitudegitude
- `pp_open` : valeur de production primaire

## 1. Importer les bibliothèques

Dans cette étape, on charge les modules nécessaires :

- `pandas` : lire le CSV et manipuler des tableaux de données ;
- `sqlite3` : créer une petite base de données SQLite pour les requêtes SQL ;
- `geopandas` : manipuler des données spatiales ;
- `Point` : créer une géométrie de point à partir des coordonnées ;
- `statistics` : calculer moyenne et écart-type.

In [ ]:
import pandas as pd
import sqlite3
import geopandas as gpd
from shapely.geometry import Point
import statistics

### Explication

- `import pandas as pd` : importe la bibliothèque **pandas** et lui donne le raccourci `pd`.
- `import sqlite3` : active le module standard Python pour travailler avec SQLite.
- `import geopandas as gpd` : importe **GeoPandas** avec le raccourci `gpd`.
- `from shapely.geometry import Point` : importe l'objet `Point`, utile pour transformer les colongitudenes `longitudegitude` et `latitudeitude` en géométries.
- `import statistics` : importe le module standard pour des statistiques descriptives simples.

## 2. Charger le fichier CSV

On lit le fichier CSV dans un **DataFrame** pandas.  
Un DataFrame est un tableau de données avec des lignes et des colongitudenes.

In [ ]:
df = pd.read_csv("data/cci_v6_2003226_ppz_takuvik_above_45n_v2.csv")
df.head()

### Explication

- `pd.read_csv(...)` : lit un fichier CSV et le transforme en DataFrame.
- Le résultat est stocké dans la variable `df`.
- `df.head()` : affiche les **5 premières lignes** pour vérifier rapidement que le fichier est bien chargé.

## 3. Vérification rapide des données

Avant de commencer l'analyse, il est utile de vérifier :

- les noms des colongitudenes ;
- le nombre total de lignes ;
- le type de données de chaque variable.

In [ ]:
print("Colongitudenes du fichier :", list(df.columns))
print("Nombre total de lignes :", len(df))
print()
print("Types de variables :")
print(df.dtypes)

### Explication

- `df.columns` : retourne les noms des colongitudenes.
- `list(df.columns)` : transforme cette information en liste plus lisible.
- `len(df)` : donne le nombre de lignes du DataFrame.
- `df.dtypes` : indique le type de chaque colongitudene (nombre entier, décimal, texte, etc.).

## 4. Créer une base SQLite

Même si les données viennent d'un CSV, on peut les placer dans une base SQLite pour faire des requêtes SQL.  
C'est pratique pour montrer aux étudiants le lien entre tableau de données et base relatitudeionnelle.

In [ ]:
conn = sqlite3.connect("pp_database.sqlite")
df.to_sql("pp_data", conn, if_exists="replace", index=False)

print("Base SQLite créée.")
print("Nom de la table :", "pp_data")

### Explication détaillée

- `sqlite3.connect("pp_database.sqlite")`
  - crée une connexion à une base SQLite ;
  - si le fichier n'existe pas encore, Python le crée automatiquement.
- la connexion est stockée dans `conn`.
- `df.to_sql("pp_data", conn, if_exists="replace", index=False)` :
  - envoie le DataFrame `df` dans la base ;
  - crée une table nommée `pp_data` ;
  - `if_exists="replace"` : remplace la table si elle existe déjà ;
  - `index=False` : évite d'ajouter l'index pandas comme colongitudene supplémentaire.

## 5. Première requête SQL : compter le nombre de pixels

On commence par une requête SQL très simple.

In [ ]:
query = "SELECT COUNT(*) AS nb_pixels FROM pp_data;"
resultat = pd.read_sql_query(query, conn)
resultat

### Explication

- `query = "..."` : stocke la requête SQL dans une chaîne de caractères.
- `COUNT(*)` : compte toutes les lignes de la table.
- `AS nb_pixels` : renomme la colongitudene de sortie.
- `pd.read_sql_query(query, conn)` :
  - envoie la requête à la base SQLite ;
  - récupère le résultat ;
  - retourne le résultat sous forme de DataFrame.

## 6. Requête SQL : minimum et maximum de `pp_open`

In [ ]:
query = '''
SELECT
    MIN(pp_open) AS pp_min,
    MAX(pp_open) AS pp_max
FROM pp_data;
'''
pd.read_sql_query(query, conn)

### Explication

- `MIN(pp_open)` : retourne la plus petite valeur de `pp_open`.
- `MAX(pp_open)` : retourne la plus grande valeur.
- `FROM pp_data` : indique la table sur laquelle la requête est exécutée.
- Les triples guillemets `''' ... '''` permettent d'écrire une requête SQL sur plusieurs lignes.

## 7. Requête SQL : moyenne globale

In [ ]:
query = "SELECT AVG(pp_open) AS moyenne_globale FROM pp_data;"
pd.read_sql_query(query, conn)

### Explication

- `AVG(pp_open)` : calcule la moyenne de la variable `pp_open`.
- `AS moyenne_globale` : donne un nom clair à la colongitudene retournée.

## 8. Requête SQL : nombre de pixels au-dessus d'un seuil

In [ ]:
query = "SELECT COUNT(*) AS nb_pixels_sup_1000 FROM pp_data WHERE pp_open > 1000;"
pd.read_sql_query(query, conn)

### Explication

- `WHERE pp_open > 1000` : filtre les lignes et conserve seulement celles dont la valeur est supérieure à 1000.
- Ensuite, `COUNT(*)` compte combien de lignes répondent à cette condition.

## 9. Requête SQL : sélectionner des observations selongitude la latitudeitude

In [ ]:
query = "SELECT * FROM pp_data WHERE latitude > 75;"
pd.read_sql_query(query, conn).head()

### Explication

- `SELECT *` : demande toutes les colongitudenes.
- `WHERE latitude > 75` : filtre les observations situées au nord de 75°N.
- `.head()` : affiche seulement les premières lignes du résultat.

## 10. Transformer le tableau en données spatiales

Le CSV contient des coordonnées `longitudegitude` et `latitudeitude`, mais ce ne sont pas encore de vraies géométries spatiales.  
On doit créer un point pour chaque ligne.

In [ ]:
geometry = [Point(xy) for xy in zip(df["longitude"], df["latitude"])]

gdf = gpd.GeoDataFrame(
    df,
    geometry=geometry,
    crs="EPSG:4326"
)

gdf.head()

### Explication détaillée

- `zip(df["longitude"], df["latitude"])` :
  - associe la longitudegitude et la latitudeitude de chaque ligne ;
  - produit des paires du type `(longitude, latitude)`.
- `Point(xy)` :
  - transforme chaque paire en géométrie de type point.
- `[Point(xy) for xy in ...]` :
  - construit une liste de points pour toutes les lignes.
- `gpd.GeoDataFrame(...)` :
  - crée un **GeoDataFrame**, c'est-à-dire un DataFrame avec une colongitudene géométrique.
- `crs="EPSG:4326"` :
  - précise le système de coordonnées ;
  - ici, il s'agit du système géographique classique en latitudeitude/longitudegitude (WGS84).

## 11. Lire le shapefile de la région

On charge ensuite le shapefile qui représente la région d'intérêt.

In [ ]:
region = gpd.read_file("data/WDPA_WDOECM_Feb2025_Public_555637925_shp-polygons.shp")
region = region.to_crs(gdf.crs)

print("Nombre de polygones dans le shapefile :", len(region))
print()
print("Colongitudenes disponibles dans le shapefile :")
print(list(region.columns))
region.head()

### Explication détaillée

- `gpd.read_file(...)` :
  - lit un fichier spatial vectoriel ;
  - ici, un shapefile.
- `region.to_crs(gdf.crs)` :
  - reprojette le shapefile dans le même système de coordonnées que les points ;
  - c'est une étape essentielle avant une opération spatiale.
- `len(region)` :
  - donne le nombre d'entités polygonales dans le shapefile.
- `region.columns` :
  - permet d'identifier les attributs disponibles ;
  - utile si on souhaite plus tard sélectionner une seule aire protégée.

## 12. Option pédagogique : vérifier si le shapefile contient plusieurs zones

Dans de nombreux cas, le shapefile WDPA contient **plusieurs polygones**.  
Si on ne filtre pas, la jointure spatiale sélectionnera les points présents dans **n'importe quel polygone** du fichier.

Si vous souhaitez travailler sur **une seule zone protégée**, il faut d'abord l'identifier à partir d'un attribut.

In [ ]:
# Exemple exploratoire : regarder quelques attributs possibles
region.head(3)

### Commentaire pédagogique

Avant de filtrer une zone unique, il faut examiner les colonnes du shapefile.  
Par exemple, si une colonne comme `NAME`, `ORIG_NAME` ou `WDPAID` existe, on peut s'en servir pour choisir une seule entité.

Exemple de logique possible (à adapter au vrai nom de colonne) :

```python
region_unique = region[region["NAME"] == "Nom_de_la_zone"]
```

Dans le notebook corrigé ci-dessous, on conserve **tous les polygones du shapefile** pour rester simple.

## 13. Extraire les points situés dans la région

On réalise une jointure spatiale entre les points et les polygones.

In [ ]:
points_region = gpd.sjoin(gdf, region, predicate="within")
points_region.head()

### Explication détaillée

- `gpd.sjoin(...)` : effectue une **jointure spatiale**.
- `gdf` : couche des points.
- `region` : couche polygonale.
- `predicate="within"` :
  - conserve les points situés **à l'intérieur** des polygones.
- Le résultat `points_region` contient seulement les observations dont la géométrie tombe dans la région.

## 14. Compter le nombre de points retenus

In [ ]:
print("Nombre de pixels dans la région :", len(points_region))

### Explication

- `len(points_region)` : compte combien de points sont restés après la sélection spatiale.

## 15. Calculer les statistiques descriptives dans la région

On veut maintenant calculer :

- la moyenne ;
- l'écart-type ;
- le minimum ;
- le maximum.

On travaille sur la colongitudene `pp_open`.

In [ ]:
valeurs = points_region["pp_open"].dropna().tolist()

moyenne_region = statistics.mean(valeurs)
ecart_type_region = statistics.stdev(valeurs)
minimum_region = min(valeurs)
maximum_region = max(valeurs)

print("Moyenne régionale :", moyenne_region)
print("Écart-type régional :", ecart_type_region)
print("Minimum régional :", minimum_region)
print("Maximum régional :", maximum_region)

### Explication détaillée

- `points_region["pp_open"]` : sélectionne la colongitudene `pp_open`.
- `.dropna()` : retire les valeurs manquantes.
- `.tolist()` : transforme la série pandas en liste Python.
- `statistics.mean(valeurs)` : calcule la moyenne.
- `statistics.stdev(valeurs)` : calcule l'écart-type.
- `min(valeurs)` et `max(valeurs)` : retournent les valeurs extrêmes.

## 16. Comparer la moyenne globale et la moyenne régionale

Cette comparaison est utile pour l'interprétation finale du laboratoire.

In [ ]:
moyenne_globale = df["pp_open"].dropna().mean()

print("Moyenne globale :", moyenne_globale)
print("Moyenne régionale :", moyenne_region)
print("Différence région - global :", moyenne_region - moyenne_globale)

### Explication

- `df["pp_open"].dropna().mean()` :
  - sélectionne la variable ;
  - retire les valeurs manquantes ;
  - calcule la moyenne avec pandas.
- la dernière ligne montre si la région est au-dessus ou au-dessous de la moyenne globale.

## 17. Interprétation possible à discuter avec les étudiants

Questions pertinentes :

1. La région présente-t-elle une production primaire plus élevée ou plus faible que l'ensemble des données ?
2. L'écart-type est-il élevé ? Cela suggère-t-il une forte variabilité spatiale ?
3. Le minimum et le maximum montrent-ils une dispersion importante ?
4. Le résultat dépend-il du choix de la région et du fait que le shapefile contienne une ou plusieurs zones ?

## 18. Fermer la connexion SQLite

In [ ]:
conn.close()
print("Connexion SQLite fermée.")

### Pourquoi fermer la connexion ?

- C'est une bonne pratique ;
- cela libère les ressources associées à la base de données ;
- cela montre aux étudiants qu'une connexion doit être ouverte puis fermée proprement.